# Live Processing Gallery (DB-backed)
This notebook uses `run_stats.sqlite3` rows as the source of truth and then renders matching images from run output folders.


In [ ]:
import sqlite3
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML, Image

def detect_repo_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/Users/openteams/Feather_Molt_Project')]
    for c in candidates:
        if (c / 'data' / 'runs').exists():
            return c
    return Path.cwd()

REPO_ROOT = detect_repo_root()
RUNS_DIR = REPO_ROOT / 'data' / 'runs'

def auto_run_id() -> str:
    run_dirs = [d for d in RUNS_DIR.iterdir() if d.is_dir()] if RUNS_DIR.exists() else []
    if not run_dirs:
        return ''
    run_dirs.sort(key=lambda d: d.stat().st_mtime, reverse=True)
    return run_dirs[0].name

RUN_ID = auto_run_id()

print('REPO_ROOT =', REPO_ROOT)
print('RUN_ID =', RUN_ID if RUN_ID else '<none>')

def fetch_gallery_rows(run_id: str):
    db_path = RUNS_DIR / run_id / 'pipeline_metrics' / 'run_stats.sqlite3'
    if not db_path.exists():
        return []
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()
    rows = cur.execute('''
        SELECT image_path, ts, feathers_saved, vlm_score, bird_id, date_text
        FROM image_stats
        WHERE run_id=? AND success=1
        ORDER BY ts DESC
    ''', (run_id,)).fetchall()
    conn.close()
    return rows

rows = fetch_gallery_rows(RUN_ID)
print('Recent successful rows:', len(rows))


In [ ]:
# Re-run this cell anytime to refresh from DB
rows = fetch_gallery_rows(RUN_ID)

def render_one(index=0):
    if not rows:
        display(HTML('<h3>No DB rows yet for this run.</h3>'))
        return

    idx = min(index, len(rows) - 1)
    r = rows[idx]
    stem = Path(r['image_path']).stem
    bird = r['bird_id'] or 'UNKNOWN'
    date_text = r['date_text'] or 'UNKNOWN'
    out_dir = RUNS_DIR / RUN_ID / 'processed'

    bbox = out_dir / f"{stem}_BoundingBoxes.jpg"
    feathers = sorted(out_dir.glob(f"{stem}_{bird}_{date_text}_Feather_*.jpg"))

    html = "<div style='background-color:#f8f9fa;padding:15px;border-radius:8px;margin-bottom:20px;'>"
    html += f"<h2>Source: {Path(r['image_path']).name}</h2>"
    html += f"<p>ts=<b>{r['ts']}</b> feathers_saved=<b>{r['feathers_saved']}</b> vlm_score=<b>{r['vlm_score']}</b></p>"
    html += "</div>"
    display(HTML(html))

    if bbox.exists():
        display(Image(filename=str(bbox), width=420))

    if not feathers:
        display(HTML('<p><i>No feather files found for this DB row metadata yet.</i></p>'))
        return

    for f in feathers:
        display(Image(filename=str(f), width=260))

if rows:
    widgets.interact(render_one, index=widgets.IntSlider(min=0, max=len(rows)-1, step=1, value=0, description='Browse:'))
else:
    print('No rows yet; wait for pipeline progress and rerun this cell.')
